# 01. 합성 시뮬레이션 분포 설계

이 노트북은 다품종 조달 5PL 합성 인스턴스의 변수, 생성규칙, 보정 근거를 명시한다.
ComprasNet은 **입력분포와 공급자–품목 구조를 보정하는 자료**로만 사용한다. WTP, WTA,
실제 공급용량, 배송비, 인센티브 반응계수와 All-or-Nothing 여부는 관측되지 않으므로
시나리오 또는 문헌·가정으로 분리한다. 여기서는 원본 대용량 CSV를 읽지 않는다.

In [1]:
# 프로젝트 루트를 찾아 상대경로와 로컬 모듈 import를 일관되게 사용한다.
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾을 수 없습니다. 저장소 안에서 노트북을 실행하세요.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEED = 20260826
PROJECT_ROOT

WindowsPath('D:/CODE/proj2/5pl')

In [2]:
# 설계변수와 생성규칙, 근거 유형을 한 표에 정리한다.
import pandas as pd

design_rows = [
    ("num_buyers", "초기 구매자 수", "고정 또는 규모 시나리오(30/50/100); smoke는 8", "민감도 분석"),
    ("num_sellers", "초기 공급자 수", "고정 또는 규모 시나리오(20/30/50); smoke는 10", "민감도 분석"),
    ("num_skus", "분석 품목군 수", "보정 대상 품목군을 선정한 뒤 고정; smoke는 5", "민감도 분석"),
    ("buyer_sku_count", "구매자별 주문 품목 수", "조달 건별 임시 품목군 수의 경험분포 또는 범주분포", "ComprasNet 보정"),
    ("buyer_sku_set", "구매자 주문 품목 조합", "품목 빈도·동시출현 가중 비복원 추출; 묶음은 AON 증거가 아님", "ComprasNet 보정"),
    ("min_qty", "주문 성립 최소수량", "max_qty의 비율 alpha; alpha는 실험수준으로 둠", "문헌·가정"),
    ("max_qty", "품목별 최대 주문수량", "동질 품목군의 양수 수량 경험분포/후보분포", "ComprasNet 보정"),
    ("seller_sku_set", "공급자별 취급 품목", "관측 참여 이분 네트워크를 재표본화; 최소 1품목 보정", "ComprasNet 보정"),
    ("seller_coverage", "공급자 품목 커버리지", "공급자별 고유 임시 품목군 수 또는 Low/Medium/High", "ComprasNet 보정"),
    ("capacity", "공급 가능 수량", "Gamma 원시값 후 품목별 목표 공급/수요비로 스케일링", "문헌·가정"),
    ("wtp", "구매자 단위 WTP", "기준가격 × (1 + 구매자 프리미엄); 관측값으로 해석 금지", "문헌·가정"),
    ("wta", "공급자 단위 WTA", "기준가격 × (1 + 공급자 가격편차); 낙찰단가와 동일하지 않음", "문헌·가정"),
    ("reference_price", "품목군 기준가격", "동질 상품 품목군의 양수 단가 중앙값/절사평균", "ComprasNet 보정"),
    ("delivery_cost", "허브→구매자 배송비", "지역 Near/Medium/Far 고정비; 자료에서 관측 불가", "문헌·가정"),
    ("region", "구매자·공급자 지역", "관측 UF 빈도 또는 통제된 지역 시나리오", "ComprasNet 보정"),
    ("supply_ratio", "최대수요 대비 공급량", "부족 0.7 / 균형 1.0 / 풍부 1.3", "민감도 분석"),
    ("price_dispersion", "공급자 가격분산", "기준가격 대비 ±5% / ±10% / ±20%", "민감도 분석"),
    ("initial_history", "초기 참여·낙찰이력", "관측 과거 참여/낙찰 횟수의 정규화값 또는 Beta 초기값", "ComprasNet 보정"),
    ("retention_parameter", "재참여·인센티브 반응계수", "모형 파라미터로 두고 범위별 민감도 분석", "민감도 분석"),
    ("all_or_nothing", "주문 전체 수락 조건", "데이터에서 추론하지 않고 실험 처리로 명시", "문헌·가정"),
    ("seed", "실험 난수 시드", "설정 파일에 고정하고 모든 난수생성기에 전달", "민감도 분석"),
]
design = pd.DataFrame(design_rows, columns=["변수", "변수 의미", "설정 또는 생성 방법", "근거 유형"])
design

,변수,변수 의미,설정 또는 생성 방법,근거 유형
0,num_buyers,초기 구매자 수,고정 또는 규모 시나리오(30/50/100); smoke는 8,민감도 분석
1,num_sellers,초기 공급자 수,고정 또는 규모 시나리오(20/30/50); smoke는 10,민감도 분석
2,num_skus,분석 품목군 수,보정 대상 품목군을 선정한 뒤 고정; smoke는 5,민감도 분석
3,buyer_sku_count,구매자별 주문 품목 수,조달 건별 임시 품목군 수의 경험분포 또는 범주분포,ComprasNet 보정
4,buyer_sku_set,구매자 주문 품목 조합,품목 빈도·동시출현 가중 비복원 추출; 묶음은 AON 증거가 아님,ComprasNet 보정
5,min_qty,주문 성립 최소수량,max_qty의 비율 alpha; alpha는 실험수준으로 둠,문헌·가정
6,max_qty,품목별 최대 주문수량,동질 품목군의 양수 수량 경험분포/후보분포,ComprasNet 보정
7,seller_sku_set,공급자별 취급 품목,관측 참여 이분 네트워크를 재표본화; 최소 1품목 보정,ComprasNet 보정
8,seller_coverage,공급자 품목 커버리지,공급자별 고유 임시 품목군 수 또는 Low/Medium/High,ComprasNet 보정
9,capacity,공급 가능 수량,Gamma 원시값 후 품목별 목표 공급/수요비로 스케일링,문헌·가정


## 생성 순서와 구조 보정

1. 기간별 구매자 주문은 외생적으로 생성한다.
2. 구매자별 품목 수와 품목 조합을 뽑고 `min_qty ≤ max_qty`가 되게 수량을 생성한다.
3. 공급자–품목 연결을 생성한 뒤 모든 공급자가 한 품목 이상, 모든 품목이 한 공급자 이상을 갖도록 보정한다.
4. 공급 원시값을 만든 후 품목별 총공급/총최대수요가 시나리오 비율에 맞도록 스케일링한다.
5. 기준가격 주변에서 WTP/WTA를 별도로 만들고 지역별 배송비와 초기 이력을 부여한다.

`Código Item Compra`는 개별 조달 품목 행 ID이며 SKU가 아니다. 정규화된 설명도 탐색적
**임시 품목군**일 뿐 동일 SKU 확정값이 아니다. 같은 조달 건의 동시출현 역시
All-or-Nothing 주문의 증거가 아니다.

In [3]:
# smoke 설정을 읽어 설계표의 현재 구현값을 확인한다.
from src.config import load_config
config = load_config("configs/smoke.json")
pd.Series(config, name="smoke 설정")

instance_id                                                  smoke_seed_20260826
seed                                                                    20260826
period                                                                         1
num_buyers                                                                     8
num_sellers                                                                   10
num_skus                                                                       5
buyer_sku_count_probs                                            [0.5, 0.3, 0.2]
buyer_sku_count_values                                                 [1, 2, 3]
seller_coverage_probability                                                  0.5
order_quantity                    {'min': 5, 'max': 30, 'minimum_fraction': 0.6}
reference_price                                      {'min': 50.0, 'max': 150.0}
buyer_price_premium                                   {'min': 0.05, 'max': 0.25}
seller_price_dispersion     

## 시나리오 그리드와 아직 고정하지 않는 값

기본 실험은 공급 부족·균형·풍부와 가격분산 Low·Medium·High의 교차설계를 사용한다.
WTP/WTA 프리미엄, 공급용량, 배송비, 잔존함수 계수, AON 여부는 ComprasNet의 직접 관측값으로
채우지 않는다. 05 노트북에서 반복거래와 공급자 수가 충분한 상품 품목군 후보를 찾은 뒤
기간과 품목군을 확정해야 수량·단가 후보분포를 보정할 수 있다.